# Pipeline Artifacts

Este notebook sustituye la logica operativa larga del baseline por una llamada unica al pipeline modular de `src/`.

Objetivo:
- reconstruir los artefactos base
- validar que la carga, forecast y bottlenecks salen correctamente
- dejar listas las tablas que despues consumiran otros notebooks o un dashboard


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import run_baseline_pipeline

PATH_XLSX = project_root / "data" / "raw" / "DatosBusquedaAvanzada20250208.xlsx"

artifacts = run_baseline_pipeline(
    workbook_path=PATH_XLSX,
    rate_window_days=60,
    dashboard_horizon_days=5,
    scenario_horizons=(5, 10, 20),
)

bm = artifacts["bm"]
forecast_dashboard = artifacts["forecast_dashboard"]
forecast_dashboard_export = artifacts["forecast_dashboard_export"]
forecast_scenarios = artifacts["forecast_scenarios"]
bottleneck_thresholds = artifacts["bottleneck_thresholds"]
task_alerts = artifacts["task_alerts"]
owner_bottlenecks = artifacts["owner_bottlenecks"]
column_bottlenecks = artifacts["column_bottlenecks"]
type_bottlenecks = artifacts["type_bottlenecks"]

sorted(artifacts.keys())


['arrivals',
 'arrivals_daily',
 'bm',
 'bottleneck_thresholds',
 'column_bottlenecks',
 'completions',
 'completions_daily',
 'forecast_base',
 'forecast_dashboard',
 'forecast_dashboard_export',
 'forecast_reference_date',
 'forecast_scenarios',
 'links',
 'open_tasks',
 'owner_bottlenecks',
 'reference_date',
 'subtasks',
 'task_alerts',
 'type_bottlenecks']

In [2]:
pd.Series({
    "snapshot_reference_date": artifacts["reference_date"],
    "forecast_reference_date": artifacts["forecast_reference_date"],
    "n_rows": len(bm),
    "n_closed": int(bm["is_closed"].sum()),
    "n_open": int((~bm["is_closed"]).sum()),
    "forecast_owners": int(forecast_dashboard.shape[0]),
    "open_alerts": int(task_alerts.shape[0]),
})


snapshot_reference_date    2026-02-08 01:16:27
forecast_reference_date    2026-02-08 00:00:00
n_rows                                    1406
n_closed                                  1359
n_open                                      47
forecast_owners                             20
open_alerts                                 47
dtype: object

In [3]:
display(forecast_dashboard_export.head(10))
display(task_alerts.head(10))
display(type_bottlenecks)


,Owner,arrival_rate_per_day,completion_rate_per_day,current_wip,expected_arrivals,expected_completions,forecast_wip,status,status_reason
0,Consultor 2,0.02,0.00,3,0.08,0.00,3.08,no_throughput,open work but no recent completions
1,Comercial 2,0.00,0.00,1,0.00,0.00,1.0,no_throughput,open work but no recent completions
2,Consultor 1,0.03,0.08,8,0.17,0.42,7.75,risk,projected backlog exceeds horizon
3,Consultor 4,0.00,0.13,6,0.00,0.67,5.33,risk,projected backlog exceeds horizon
4,Técnico 8,0.05,0.12,5,0.25,0.58,4.67,risk,projected backlog exceeds horizon
5,Técnico 4,0.03,0.17,5,0.17,0.83,4.33,risk,projected backlog exceeds horizon
6,Otros 2,0.02,0.03,4,0.08,0.17,3.92,risk,projected backlog exceeds horizon
7,Técnico 7,0.05,0.10,4,0.25,0.50,3.75,risk,projected backlog exceeds horizon
8,Técnico 10,0.00,0.02,2,0.00,0.08,1.92,risk,projected backlog exceeds horizon
9,Consultor 6,0.00,0.05,2,0.00,0.25,1.75,risk,projected backlog exceeds horizon


,Card ID,Owner,Column Name,task_status,Type Name,age_days,days_since_last_moved,subtasks_count,has_any_links,owner_forecast_wip,...,owner_forecast_status_reason,benchmark_source,benchmark_median_days,benchmark_p90_days,age_vs_benchmark,blocked_state,block_time_hours,alert_score,alert_level,alert_reason
0,5449,Otros 2,Backlog,open,PROYECTO INTERNO,306.49,243.0,5.0,1,3.92,...,projected backlog exceeds horizon,owner_type,30.98,104.24,9.89,yes,6479.90,11,bottleneck,old_open; stagnant; older_than_history; depend...
1,4480,Otros 2,Backlog,open,PROYECTO INTERNO,331.51,243.0,6.0,1,3.92,...,projected backlog exceeds horizon,owner_type,30.98,104.24,10.70,no,0.00,8,bottleneck,old_open; stagnant; older_than_history; depend...
2,4807,Consultor 1,En progreso,in_progress,PROYECTO INTERNO,326.33,243.0,3.0,1,7.75,...,projected backlog exceeds horizon,owner_type,132.52,192.50,2.46,no,0.00,8,bottleneck,old_open; stagnant; older_than_history; depend...
3,4876,Consultor 6,En progreso,in_progress,PROYECTO EXTERNO,324.47,277.0,0.0,0,1.75,...,projected backlog exceeds horizon,owner_type,18.19,189.59,17.84,yes,2303.93,8,bottleneck,old_open; stagnant; older_than_history; curren...
4,6360,Consultor 4,Backlog,open,PRODUCTO,270.70,243.0,3.0,1,5.33,...,projected backlog exceeds horizon,owner_type,34.53,85.98,7.84,no,0.00,8,bottleneck,old_open; stagnant; older_than_history; depend...
5,5108,Técnico 8,En progreso,in_progress,PROYECTO EXTERNO,317.71,115.0,0.0,0,4.67,...,projected backlog exceeds horizon,owner_type,2.15,33.43,147.52,yes,2763.35,6,bottleneck,old_open; older_than_history; currently_blocke...
6,5191,Consultor 4,En progreso,in_progress,PROYECTO EXTERNO,317.27,118.0,0.0,0,5.33,...,projected backlog exceeds horizon,owner_type,21.83,98.73,14.53,yes,2564.17,6,bottleneck,old_open; older_than_history; currently_blocke...
7,5810,Consultor 2,Backlog,open,PROYECTO INTERNO,299.45,242.0,0.0,1,3.08,...,open work but no recent completions,type,4.01,14.78,74.72,no,0.00,6,bottleneck,stagnant; older_than_history; dependency_risk;...
8,5809,Consultor 2,Backlog,open,PROYECTO INTERNO,299.45,242.0,0.0,1,3.08,...,open work but no recent completions,type,4.01,14.78,74.72,no,0.00,6,bottleneck,stagnant; older_than_history; dependency_risk;...
9,7140,Técnico 7,Backlog,open,PRODUCTO,247.36,131.0,0.0,1,3.75,...,projected backlog exceeds horizon,owner_type,13.07,48.37,18.93,no,2662.69,6,bottleneck,old_open; older_than_history; dependency_risk;...


,type_name,open_tasks,risk_tasks,bottleneck_tasks,median_age_days,max_age_days,median_days_since_last_moved,mean_age_vs_benchmark,dependency_risk_tasks,complexity_risk_tasks,currently_blocked_tasks,type_open_age_p75,type_open_age_p90,bottleneck_status
0,PROYECTO INTERNO,16,3,8,212.52,331.51,242.0,37.44,5,4,3,301.21,322.44,bottleneck
1,PROYECTO EXTERNO,15,6,8,155.67,324.47,116.5,38.46,0,0,7,263.60,317.54,bottleneck
2,PRODUCTO,16,8,4,84.70,270.70,79.0,8.55,4,1,0,139.43,247.36,bottleneck
